# Chapter 10 lab — Which still-running worker may write?

Draft educator companion v2 · 8 September 2026 · 90 minutes. This version supersedes v1 for new classes; retain your old attempts.

**What we are building:** a useful agent for Lucy’s shop, whose actual Python boundaries can be investigated and repaired. **What Lucy loses if this boundary fails:** A worker from an earlier authority epoch can append evidence to Lucy’s current work.

Read [the chapter](https://www.profrod.ai/book/ch10-worker-recovery). Prerequisites: Chapter 9 effects; generation numbers, leases and comparisons at the write boundary. Use the locked Python 3.14 checkout named by the educator companion. Set SOVEREIGN_AGENT_REPO if the notebook is outside that checkout. No packages are installed here; no model or channel credentials are needed.

Today’s sequence: predict 10 minutes; decision warm-up 10; real-code break and repair 45; transfer 15; exit ticket 10. The warm-up is deliberately small. The central task edits a temporary copy of the actual implementation, observes its effect, repairs it, and traces a source line to a retained result.

**Execution preparation:** One bounded Python subprocess per trial; temporary SQLite state; no paid model or channel call. No worker is killed. Allow up to 60 seconds per trial for a cold machine; measure elapsed classroom time yourself. Copied Python is not a security sandbox. Run only these reviewed local exercises, never arbitrary downloaded code. Close the lab at the end to remove its temporary copy; save your source patch and evidence first.


## Predict before running

Worker A is still alive after its lease expires. Worker B claims generation two. Should A’s process liveness allow it to append a transcript or send an order?

Write a prediction for the real mutation too: or work.epoch != current_epoch → or False. Name an observation that would disprove your explanation.


In [ ]:
import copy
import hashlib
import json
import os
import runpy
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 14):
    raise RuntimeError("Use the book Python 3.14 environment for Chapters 2–16.")
# Open the notebook inside your source checkout, or set this path explicitly.
start = Path(os.environ.get("SOVEREIGN_AGENT_REPO", Path.cwd())).resolve()
ROOT = next(
    (p for p in (start, *start.parents) if (p / "book/always_on/checkpoints/ch10.py").is_file()),
    None,
)
if ROOT is None:
    raise RuntimeError("Set SOVEREIGN_AGENT_REPO to the Sovereign Agent checkout.")
CHECKPOINT = ROOT / "book/always_on/checkpoints/ch10.py"
EXPECTED_CHECKPOINT_SHA256 = "f4fd29c6acba491b3348e71c9b95e9dc3c26d7f890dc9181c7bce74a840585ed"
if hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest() != EXPECTED_CHECKPOINT_SHA256:
    raise RuntimeError(
        "Checkpoint differs; use the exact cohort commit and versioned files in the companion."
    )
print("Chapter 10 checkpoint bytes match this lesson. No model or channel has been called.")

lab_support = runpy.run_path(str(ROOT / "book/always_on/educator/runtime_labs_v1.py"))
RuntimeLab = lab_support["RuntimeLab"]

## Ten-minute decision warm-up

Implement decide(case). Compare presented and current dictionaries on work, owner, generation and epoch. Return CURRENT only if all match, current.status is RUNNING, cancelled is false and now < expires; otherwise STALE. Inputs are trusted typed records. Equality with expiry is already expired.

Visible tests are a specification, not secret examination questions. A lookup table of CASES can pass them. Your teacher assesses your implementation, a novel teacher-chosen case, and the actual runtime repair. Do not infer mastery from the worked answer or from running supplied code. Input mutation is a failure even when expected and observed values match.


In [ ]:
def grade(candidate, cases):
    results = []
    for index, (case, expected) in enumerate(cases, 1):
        supplied = copy.deepcopy(case)
        row = {"case": index, "expected": expected, "mutated": False}
        try:
            observed = candidate(supplied)
            row["mutated"] = supplied != case
            try:
                encoded = json.dumps(observed, allow_nan=False, sort_keys=True)
                safe = json.loads(encoded)
            except (TypeError, ValueError, OverflowError, RecursionError):
                row.update(
                    status="FAILED",
                    reason="unsupported_return_type",
                    observed={
                        "result_type": type(observed).__name__,
                        "reason": "unsupported_return_type",
                    },
                )
            else:
                same = encoded == json.dumps(expected, allow_nan=False, sort_keys=True)
                row.update(
                    status="PASS" if same and not row["mutated"] else "FAILED",
                    reason=(
                        "input_mutated"
                        if row["mutated"]
                        else "matched_contract"
                        if same
                        else "wrong_result"
                    ),
                    observed=safe,
                )
        except NotImplementedError:
            row.update(status="NOT_SUBMITTED", reason="not_submitted", mutated=supplied != case)
        except Exception as error:
            row.update(
                status="FAILED",
                reason="candidate_exception",
                error_type=type(error).__name__,
                mutated=supplied != case,
            )
        results.append(row)
    return results


def assessment_status(results):
    counts = {
        name: sum(row["status"] == name for row in results)
        for name in ("PASS", "FAILED", "NOT_SUBMITTED")
    }
    status = (
        "NOT_SUBMITTED"
        if not results or counts["NOT_SUBMITTED"] == len(results)
        else "PARTIAL"
        if counts["NOT_SUBMITTED"]
        else "FAILED"
        if counts["FAILED"]
        else "PASSED_VISIBLE_CONTRACT"
    )
    return {
        "status": status,
        "counts": counts,
        "scope": "Visible cases need teacher review of code, novel cases and runtime evidence.",
    }


def decide(case):
    raise NotImplementedError("Write your function before consulting the worked solution.")

In [ ]:
CASES = [
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 1,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "CURRENT",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "B",
                "generation": 2,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 1,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 100,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 1,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": True,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "B",
                "generation": 1,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 2,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 1,
                "epoch": 2,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "afternoon",
                "owner": "A",
                "generation": 1,
                "epoch": 1,
                "status": "RUNNING",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 1,
                "epoch": 1,
                "status": "DONE",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
    (
        {
            "current": {
                "work": "morning",
                "owner": "A",
                "generation": 1,
                "epoch": 1,
                "status": "READY",
                "cancelled": False,
                "expires": 100,
            },
            "presented": {"work": "morning", "owner": "A", "generation": 1, "epoch": 1},
            "now": 99,
        },
        "STALE",
    ),
]
submission_results = grade(decide, CASES)
submission_summary = assessment_status(submission_results)
print(json.dumps({"summary": submission_summary, "cases": submission_results}, indent=2))

## Forty-five-minute centre — break and repair the real implementation

Target `src/sovereign_agent/assistant_work.py` at the line printed below. Read the surrounding function and the probe before executing it. The helper copies actual source, learner code, checkpoints and skills into a new temporary directory. It verifies the original file fingerprint and never edits the checkout.

The probe is plain Python, visible at `lab.probe`; open it and follow its inputs into the target function. It uses real tool dispatch or database operations, not a second toy implementation. The literal expected observations were authored separately from the code under test.

This controlled state injection changes both epoch representations while holding owner, generation and lease fixed. An ordinary replacement changes several fields and would conceal a missing epoch check. The full checkpoint separately includes real process death.

Record baseline and broken observations. Before revealing the answer, replace REPAIR_FRAGMENT with your repair of the marked fragment, or edit `lab.target` directly and set EDITED_COPY=True. A blank repair remains NOT_SUBMITTED. Do not edit installed runtime code, the original checkout, or the printed expected answers.


In [ ]:
lab = RuntimeLab(ROOT, 10)
print(str(lab.target))
print(lab.source_excerpt())
print(lab.probe.read_text())
baseline_record = lab.run("BASELINE_REFERENCE", expected=lab.spec["expected_baseline"])
lab.break_source()
print(lab.source_excerpt())
broken_record = lab.run("BROKEN_REFERENCE", expected=lab.spec["expected_broken"])
# PASS here means the supplied fault produced the expected failure, not that it is safe.
assert baseline_record["status"] == "PASS", "Inspect retained baseline diagnostics"
assert broken_record["status"] == "PASS", "Inspect retained fault diagnostics"

In [ ]:
REPAIR_FRAGMENT = None  # TODO: replacement text for the marked broken fragment.
EDITED_COPY = False  # Set True only after saving your own edit to lab.target.
if REPAIR_FRAGMENT is not None:
    lab.repair(REPAIR_FRAGMENT)
if REPAIR_FRAGMENT is not None or EDITED_COPY:
    student_repair = lab.run("STUDENT_REPAIR", expected=lab.spec["expected_baseline"])
    student_source = lab.target.read_text()
else:
    student_repair = {"status": "NOT_SUBMITTED"}
    student_source = None
TRACE = {}  # TODO: source_path, source_line, observation_key, observed_value, explanation.
trace_result = lab.trace(TRACE, broken_record)
print(json.dumps({"repair": student_repair, "trace": trace_result}, indent=2))

## Transfer — change one constraint

Create one case for a changed database epoch with unchanged owner and generation, and one exactly at lease expiry. Then find the real stale-worker checks at transcript, completion and supplier boundaries.

Add independently calculated (input, expected) pairs below. For the runtime extension, edit only the copied probe and retain a new trial. Do not feed runtime objects into a pure-function case.

**Real-code extension:** Change only generation in another fresh trial. Then change only owner, work ID and status in separate cases. Explain why liveness does not answer any of these questions.


In [ ]:
TRANSFER_CASES = []
transfer_results = grade(decide, TRANSFER_CASES)
transfer_summary = assessment_status(transfer_results)
print(json.dumps({"summary": transfer_summary, "cases": transfer_results}, indent=2))

## Optional whole-chapter checkpoint

This separate reference retains the complete integration scenario. It is not your repair grade. Set RUN_FULL_CHECKPOINT=True only after the central experiment and with time to inspect it. Chapter 10 and 16 include real local SIGKILL; Chapters 9–10 and 15–16 start supplier processes; Chapter 11 starts MCP; Chapter 15 never installs systemd. Failures print stdout and stderr before any assertion.


In [ ]:
RUN_FULL_CHECKPOINT = False
if RUN_FULL_CHECKPOINT:
    # This supplied cumulative program is separate from grading your function.
    # It uses fixture models/channels. Some chapters start local child processes.
    # No --live, --telegram or --containers switch is added.
    reference_environment = {
        k: v
        for k, v in os.environ.items()
        if k in {"PATH", "SYSTEMROOT", "TMPDIR", "LANG", "LC_ALL"}
    }
    reference_environment["PYTHONPATH"] = str(ROOT / "src")
    reference_run = subprocess.run(
        [sys.executable, str(CHECKPOINT)],
        cwd=ROOT,
        env=reference_environment,
        capture_output=True,
        text=True,
        timeout=180,
        check=False,
    )
    print(reference_run.stdout)
    print(reference_run.stderr, file=sys.stderr)
    assert reference_run.returncode == 0, "Checkpoint failed; stdout and stderr are retained above"
    EXPECTED_OBSERVATIONS = ["Live stale worker refused: 3", "New model calls during recovery: 0"]
    assert all(text in reference_run.stdout for text in EXPECTED_OBSERVATIONS)
    print("REFERENCE_CHECKPOINT_PASSED — this is not your submission grade.")

## Worked answers — reveal after retaining your attempt

Both cases are STALE. The checkpoint keeps one old worker alive and independently kills another; replacements use the existing approved operation with zero new model calls. Checking once at task start would leave a stale write window.

This controlled state injection changes both epoch representations while holding owner, generation and lease fixed. An ordinary replacement changes several fields and would conceal a missing epoch check. The full checkpoint separately includes real process death.

Teacher trace key: `src/sovereign_agent/assistant_work.py:349` → {'stale_epoch_write': False, 'transcript_rows': 0} before the fault and {'stale_epoch_write': True, 'transcript_rows': 1} after it. The source line changes the real probe result. Merely printing those values is not a repair.


In [ ]:
def worked_decide(case):
    current, presented = case["current"], case["presented"]
    same = all(current[key] == presented[key] for key in ("work", "owner", "generation", "epoch"))
    valid = (
        same
        and current["status"] == "RUNNING"
        and not current["cancelled"]
        and case["now"] < current["expires"]
    )
    return "CURRENT" if valid else "STALE"


worked_results = grade(worked_decide, CASES)
assert all(row["status"] == "PASS" for row in worked_results)
# A fresh copy keeps the worked repair separate from your retained source/evidence.
worked_lab = RuntimeLab(ROOT, 10)
try:
    worked_lab.break_source()
    worked_lab.repair(worked_lab.spec["before"])
    worked_repair = worked_lab.run("WORKED_REPAIR", expected=worked_lab.spec["expected_baseline"])
finally:
    worked_lab.close()
print("WORKED_EXAMPLE_PASSED; original submission remains separate.")
assert worked_repair["status"] == "PASS", "Worked repair failed; inspect diagnostics"

In [ ]:
def tempting_shortcut(case):
    return (
        "CURRENT"
        if case["presented"]["owner"] == case["current"]["owner"]
        and case["now"] < case["current"]["expires"]
        and (not case["current"]["cancelled"])
        else "STALE"
    )


shortcut_results = grade(tempting_shortcut, CASES)
assert any(row["status"] == "FAILED" for row in shortcut_results)
print(json.dumps(shortcut_results, indent=2))

## Exit ticket and evidence

Submit the first attempt, warm-up cases, your repaired source, baseline/broken/repaired records, source-to-result trace, transfer prediction and outcome. Explain Lucy’s consequence, which observation would falsify your claim, and what the probe leaves unproved.

Process liveness, current authority and external completion answer different questions. A lease is not a mechanism for undoing a call already sent. The function models a predicate; real checking and the mediated write must share the correct transactional boundary. The experiment is local, not proof of arbitrary distributed fencing.

Use the companion pilot record to capture actual completion times and errors. No automated run is evidence of a student learning gain.


In [ ]:
runtime_results = {
    "baseline": baseline_record,
    "broken": broken_record,
    "student_repair": student_repair,
    "student_source": student_source,
    "trace": trace_result,
    "worked_repair": worked_repair,
}
EVIDENCE_PATH = None  # Optional new path; existing evidence is never overwritten.
if EVIDENCE_PATH is not None:
    with Path(EVIDENCE_PATH).open("x") as stream:
        json.dump(
            {
                "submission": submission_results,
                "transfer": transfer_results,
                "runtime": runtime_results,
            },
            stream,
            indent=2,
        )
lab.close()